### Practica 2 - RA 1
**Alumno**: Gabriel Herrador
**Modulo**: M4 - Sistemas de Big Data

### Paso 1: Web Scraping de la Competencia

**Propósito**: 
El objetivo de esta fase es extraer de forma automatizada los datos de ofertas y precios de productos de la competencia a partir del archivo local `competitor_prices.html`. Como esta fuente no dispone de un formato estructurado previo, se aplica **Web Scraping** mediante la librería `BeautifulSoup` para analizar la estructura HTML, aislar las tarjetas de oferta y exportar la información tabulada a un archivo intermedio (`competitor_prices_extracted.csv`).

#### Procedimiento realizado
1. **Lectura y parseo HTML:** Se procesa el documento mediante `BeautifulSoup(f, "html.parser")` buscando los contenedores individuales de las ofertas bajo la etiqueta `<article class="offer-card">`.
2. **Extracción de variables clave:** De cada tarjeta de producto se capturan dinámicamente:
   - `offer_id`: Identificador único de la oferta (obtenido del atributo `data-offer-id`).
   - `product_id`: Identificador del producto asociado (obtenido del atributo `data-product-id`).
   - `competitor`: Nombre de la empresa o comercio competidor (`.competitor`).
   - `product_name`: Nombre comercial del producto (`.product-name`).
   - `availability`: Estado de stock o disponibilidad (`.availability`).
   - `raw_price`: Precio publicado en formato texto original con símbolos de divisa (`.price`).
3. **Persistencia intermedia:** Se consolida la información en un DataFrame de Pandas (`df_competitors`) de 31 registros y se exporta a `competitor_prices_extracted.csv` para garantizar su disponibilidad en las fases posteriores de limpieza y cruce.

In [21]:
from bs4 import BeautifulSoup
import pandas as pd

# ==========================================
# PASO 1 (DEFINITIVO): WEB SCRAPING DE LA COMPETENCIA
# ==========================================
print("--- INICIANDO WEB SCRAPING DE COMPETENCIA ---")

try:
    with open("competitor_prices.html", "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")
    print("Archivo 'competitor_prices.html' cargado correctamente.")
except FileNotFoundError:
    print("Error: No se encuentra el archivo 'competitor_prices.html'.")

ofertas_list = []

# Seleccionamos todas las tarjetas de oferta exactas
cards = soup.select("article.offer-card")
print(f"Tarjetas de oferta detectadas: {len(cards)}")

for card in cards:
    try:
        # Extraemos los campos usando los selectores correctos de la tarjeta
        offer_id = card.get("data-offer-id", "N/A")
        product_id = card.get("data-product-id", "N/A")
        
        comp_tag = card.select_one(".competitor")
        name_tag = card.select_one(".product-name")
        avail_tag = card.select_one(".availability")
        price_tag = card.select_one(".price")

        competitor = comp_tag.get_text(strip=True) if comp_tag else "N/A"
        product_name = name_tag.get_text(strip=True) if name_tag else "N/A"
        availability = avail_tag.get_text(strip=True) if avail_tag else "N/A"
        raw_price = price_tag.get_text(strip=True) if price_tag else "N/A"

        ofertas_list.append({
            "offer_id": offer_id,
            "product_id": product_id,
            "competitor": competitor,
            "product_name": product_name,
            "availability": availability,
            "raw_price": raw_price
        })
    except Exception as e:
        continue

df_competitors = pd.DataFrame(ofertas_list)

print(f"Total de ofertas reales extraídas: {len(df_competitors)}")
display(df_competitors.head())

# Guardamos el CSV definitivo
if len(df_competitors) > 0:
    df_competitors.to_csv("competitor_prices_extracted.csv", index=False, encoding="utf-8")
    print("\n¡CSV 'competitor_prices_extracted.csv' guardado correctamente!")

--- INICIANDO WEB SCRAPING DE COMPETENCIA ---
Archivo 'competitor_prices.html' cargado correctamente.
Tarjetas de oferta detectadas: 31
Total de ofertas reales extraídas: 31


,offer_id,product_id,competitor,product_name,availability,raw_price
0,O0002,1,CosmeticLab,Essence Lash Princess Mascara,Disponible,"9,49 €"
1,O0004,6,PerfumeHub,Calvin Klein CK ONE Eau de Toilette,En stock,"€ 47,90"
2,O0026,152,MegaStore,Raqueta de tenis,Disponible,"49,50 EUR"
3,O0013,91,UrbanFeet,Off White / Red Sports Sneakers,Últimas unidades,"124,90 €"
4,O0029,193,MegaStore,Fashion Gold Women's Watch,Últimas unidades,"€ 799,95"



¡CSV 'competitor_prices_extracted.csv' guardado correctamente!


### Paso 2: Análisis e Inspección por Separado de las Fuentes
**Propósito**:
Realizar un análisis exploratorio de datos (EDA) preliminar e independiente para cada una de las tres fuentes de información obtenidas (`shopdata_sales.csv`, el catálogo de la API DummyJSON y el CSV de competencia `competitor_prices_extracted.csv`). El objetivo es comprender la estructura de cada dataset, identificar el volumen de registros y variables, detectar valores nulos, verificar los tipos de datos iniciales y evaluar la calidad general de los datos antes de proceder a su limpieza.

#### Procedimiento realizado
1. **Inspección del Dataset de Ventas:** Carga del CSV histórico de transacciones para revisar dimensiones, tipos de datos y distribución de nulos.
2. **Inspección del Catálogo de la API:** Revisión de la estructura, columnas y nulos del DataFrame extraído del servidor externo.
3. **Inspección del Dataset de Competencia:** Verificación del contenido del fichero CSV intermedio generado en el Paso 1 para asegurar que el scraping se ha volcado correctamente.

In [4]:
# ==========================================
# PASO 2: ANÁLISIS E INSPECCIÓN EXPLORATORIA (EDA)
# ==========================================
import pandas as pd
import requests

print("=== 1. INSPECCIÓN: DATASET DE VENTAS (shopdata_sales.csv) ===")
df_sales = pd.read_csv("shopdata_sales.csv")
print(f"Dimensiones (Filas, Columnas): {df_sales.shape}")
print("\nTipos de datos y nulos:")
display(df_sales.info())
print("\nPrimeras filas:")
display(df_sales.head())
print("\nValores nulos por columna:")
display(df_sales.isna().sum())

print("\n" + "="*60 + "\n")

print("=== 2. INSPECCIÓN: CATÁLOGO DE LA API (DummyJSON) ===")
# Consumo rápido de la API para el EDA
response = requests.get("https://dummyjson.com/products?limit=200")
df_products = pd.DataFrame(response.json().get("products", []))

print(f"Dimensiones (Filas, Columnas): {df_products.shape}")
print("\nTipos de datos y nulos:")
display(df_products.info())
print("\nPrimeras filas:")
display(df_products.head())
print("\nValores nulos por columna:")
display(df_products.isna().sum())

print("\n" + "="*60 + "\n")

print("=== 3. INSPECCIÓN: DATOS DE COMPETENCIA (competitor_prices_extracted.csv) ===")
df_competitors = pd.read_csv("competitor_prices_extracted.csv")
print(f"Dimensiones (Filas, Columnas): {df_competitors.shape}")
print("\nTipos de datos y nulos:")
display(df_competitors.info())
print("\nPrimeras filas:")
display(df_competitors.head())
print("\nValores nulos por columna:")
display(df_competitors.isna().sum())

=== 1. INSPECCIÓN: DATASET DE VENTAS (shopdata_sales.csv) ===
Dimensiones (Filas, Columnas): (75000, 7)

Tipos de datos y nulos:
<class 'pandas.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   sale_id           75000 non-null  str    
 1   date              75000 non-null  str    
 2   product_id        75000 non-null  int64  
 3   units             75000 non-null  int64  
 4   unit_price        75000 non-null  float64
 5   customer_country  75000 non-null  str    
 6   channel           75000 non-null  str    
dtypes: float64(1), int64(2), str(4)
memory usage: 6.4 MB


None


Primeras filas:


,sale_id,date,product_id,units,unit_price,customer_country,channel
0,S0000001,2026-01-22T11:58:50,31,1,0.79,ES,app
1,S0000002,2026-08-08T15:14:19,122,1,262.30,ES,web
2,S0000003,2026-03-18T19:45:17,6,1,49.32,PT,app
3,S0000004,2026-03-19T02:20:25,121,1,193.09,DE,app
4,S0000005,2026-02-02T01:11:01,6,1,42.47,IT,web



Valores nulos por columna:


sale_id             0
date                0
product_id          0
units               0
unit_price          0
customer_country    0
channel             0
dtype: int64

... (8 outputs left out)


Primeras filas:


,offer_id,product_id,competitor,product_name,availability,raw_price
0,O0002,1,CosmeticLab,Essence Lash Princess Mascara,Disponible,"9,49 €"
1,O0004,6,PerfumeHub,Calvin Klein CK ONE Eau de Toilette,En stock,"€ 47,90"
2,O0026,152,MegaStore,Raqueta de tenis,Disponible,"49,50 EUR"
3,O0013,91,UrbanFeet,Off White / Red Sports Sneakers,Últimas unidades,"124,90 €"
4,O0029,193,MegaStore,Fashion Gold Women's Watch,Últimas unidades,"€ 799,95"



Valores nulos por columna:


offer_id        0
product_id      0
competitor      0
product_name    0
availability    0
raw_price       0
dtype: int64

### Paso 3: Limpieza, Transformación y Normalización Individual

**Propósito**: 
Aplicar un proceso de transformación independiente a cada uno de los tres datasets para estandarizar formatos, corregir tipos de datos y preparar las variables clave antes de su cruce analítico:
1. **Dataset de Ventas:** Conversión de la columna de fecha a formato `datetime` de Pandas para permitir análisis temporales.
2. **Catálogo de la API (DummyJSON):** Filtrado de las columnas esenciales (como identificador, título, categoría, precio y stock) y renombrado de `id` a `product_id` para unificar los criterios de relación.
3. **Datos de Competencia:** Limpieza de la columna `raw_price` (eliminación de símbolos de moneda como `€` o `EUR`, así como la normalización de separadores decimales) para transformarla en una variable numérica limpia (`clean_price`), asegurando la correspondencia con `product_id`.

#### Procedimiento realizado
1. **Normalización temporal (Ventas):** Se transforma la cadena de texto de la columna `date` a un objeto tipo fecha/hora estándar (`datetime64[ns]`).
2. **Selección y proyección (API):** Se seleccionan 8 columnas clave del catálogo general y se adapta el esquema renombrando los campos necesarios para el posterior cruce de datos.
3. **Parsing de precios (Competencia):** Se implementa una función de limpieza basada en expresiones y reemplazos de cadenas para extraer el valor numérico flotante puro de los precios de los competidores, gestionando posibles nulos o valores atípicos.

In [5]:
# ==========================================
# PASO 3: LIMPIEZA Y TRANSFORMACIÓN INDIVIDUAL
# ==========================================
import pandas as pd
import numpy as np


print("=== 1. LIMPIEZA: DATASET DE VENTAS ===")
# Copia para trabajar de forma limpia
df_sales_clean = df_sales.copy()
# Convertir la columna de fecha a tipo datetime de Pandas
df_sales_clean['date'] = pd.to_datetime(df_sales_clean['date'])
print("Fechas de ventas convertidas a datetime correctamente.")
display(df_sales_clean.head(2))

print("\n" + "="*60 + "\n")

print("=== 2. LIMPIEZA: CATÁLOGO DE LA API (DummyJSON) ===")
# Seleccionamos las columnas de interés para el análisis y cruce
cols_api_to_keep = ['id', 'title', 'category', 'price', 'discountPercentage', 'rating', 'stock', 'brand']
df_products_clean = df_products[cols_api_to_keep].copy()
# Renombramos 'id' a 'product_id' para facilitar el merge posterior
df_products_clean = df_products_clean.rename(columns={'id': 'product_id', 'price': 'api_price', 'title': 'api_product_name'})
print(f"Dimensiones del catálogo limpio: {df_products_clean.shape}")
display(df_products_clean.head(2))

print("\n" + "="*60 + "\n")

print("=== 3. LIMPIEZA: DATOS DE COMPETENCIA ===")
df_comp_clean = df_competitors.copy()

# Función robusta para limpiar los precios de texto (ej. "€ 47,90" -> 47.90)
def clean_price_string(price_str):
    if pd.isna(price_str):
        return np.nan
    # Eliminamos símbolos de moneda y espacios
    cleaned = str(price_str).replace('€', '').replace('EUR', '').replace('$', '').strip()
    # Cambiamos formato europeo de comas por puntos estándar
    cleaned = cleaned.replace('.', '').replace(',', '.') if ',' in cleaned and '.' in cleaned else cleaned.replace(',', '.')
    # Nos quedamos únicamente con los dígitos y el punto decimal
    cleaned = ''.join([c for c in cleaned if c.isdigit() or c == '.'])
    try:
        return float(cleaned)
    except ValueError:
        return np.nan

df_comp_clean['clean_price'] = df_comp_clean['raw_price'].apply(clean_price_string)
# Aseguramos que product_id sea numérico entero para el cruce
df_comp_clean['product_id'] = pd.to_numeric(df_comp_clean['product_id'], errors='coerce')

print("Muestra de precios limpios en competencia:")
display(df_comp_clean[['product_name', 'competitor', 'raw_price', 'clean_price']].head(5))
print(f"Valores nulos generados en la limpieza de precios: {df_comp_clean['clean_price'].isna().sum()}")

=== 1. LIMPIEZA: DATASET DE VENTAS ===
Fechas de ventas convertidas a datetime correctamente.


,sale_id,date,product_id,units,unit_price,customer_country,channel
0,S0000001,2026-01-22 11:58:50,31,1,0.79,ES,app
1,S0000002,2026-08-08 15:14:19,122,1,262.30,ES,web




=== 2. LIMPIEZA: CATÁLOGO DE LA API (DummyJSON) ===
Dimensiones del catálogo limpio: (194, 8)


,product_id,api_product_name,category,api_price,discountPercentage,rating,stock,brand
0,1,Essence Mascara Lash Princess,beauty,9.99,10.48,2.56,99,Essence
1,2,Eyeshadow Palette with Mirror,beauty,19.99,18.19,2.86,34,Glamour Beauty




=== 3. LIMPIEZA: DATOS DE COMPETENCIA ===
Muestra de precios limpios en competencia:


,product_name,competitor,raw_price,clean_price
0,Essence Lash Princess Mascara,CosmeticLab,"9,49 €",9.49
1,Calvin Klein CK ONE Eau de Toilette,PerfumeHub,"€ 47,90",47.90
2,Raqueta de tenis,MegaStore,"49,50 EUR",49.50
3,Off White / Red Sports Sneakers,UrbanFeet,"124,90 €",124.90
4,Fashion Gold Women's Watch,MegaStore,"€ 799,95",799.95


Valores nulos generados en la limpieza de precios: 0


## 5. Unificación, Enriquecimiento y Exportación a Formato Parquet

Una vez saneadas e individualizadas las fuentes de información, se procedió a consolidar todo el ecosistema de datos en una única tabla analítica mediante cruces relacionales (*merge*) apoyados en el identificador único de producto (`product_id`).

### 5.1. Integración del Histórico de Ventas y Catálogo Oficial
En una primera fase, se ejecutó un cruce interno (*inner join*) entre las 75.000 transacciones limpias y el catálogo depurado de la API (*DummyJSON*). Esta operación permitió enriquecer cada registro de venta con su categoría comercial, valoración de clientes, stock disponible y precio oficial de lista.

### 5.2. Incorporación de Inteligencia de Mercado (Competencia)
Para dotar al modelo de contexto competitivo sin perder registros históricos de aquellos artículos sin seguimiento externo, se agruparon las 31 ofertas de la competencia por producto (calculando precios mínimo, máximo, medio y volumen de ofertas) y se integraron mediante un cruce externo por la izquierda (*left join*).

### 5.3. Generación del Dataset Maestro y Persistencia en Parquet
El flujo de integración culminó con la creación de un **Dataset Maestro** unificado y tipado. Con el fin de maximizar la velocidad de lectura en futuras consultas analíticas, optimizar el almacenamiento en disco y conservar intactos los esquemas de datos, el resultado final se exportó bajo el estándar columnar **Apache Parquet** (`master_sales_market_data.parquet`).


In [6]:
# ==========================================
# PASO 4: UNIFICACIÓN Y EXPORTACIÓN A PARQUET
# ==========================================
import os
import pandas as pd

print("=== 1. CRUCE DE VENTAS CON CATÁLOGO API ===")
df_unified = pd.merge(
    df_sales_clean, 
    df_products_clean, 
    on='product_id', 
    how='inner'
)
print(f"Registros tras cruzar Ventas + Catálogo: {df_unified.shape[0]} filas x {df_unified.shape[1]} columnas")

print("\n=== 2. AGREGACIÓN Y CRUCE CON COMPETENCIA ===")
# Agrupamos por product_id para obtener métricas competitivas si hay varios vendedores
df_comp_agg = df_comp_clean.groupby('product_id').agg(
    comp_min_price=('clean_price', 'min'),
    comp_max_price=('clean_price', 'max'),
    comp_avg_price=('clean_price', 'mean'),
    comp_offers_count=('competitor', 'count')
).reset_index()

# Cruzamos con el dataframe principal mediante left join para no perder ventas sin competencia directa
df_master = pd.merge(
    df_unified,
    df_comp_agg,
    on='product_id',
    how='left'
)

print(f"Dimensiones del Dataset Maestro Final: {df_master.shape}")
display(df_master.head(3))

print("\n=== 3. EXPORTACIÓN Y VALIDACIÓN EN APACHE PARQUET ===")
parquet_file = "master_sales_market_data.parquet"
df_master.to_parquet(parquet_file, index=False, engine='pyarrow')

# Comprobación del archivo generado en disco
file_size_mb = os.path.getsize(parquet_file) / (1024 * 1024)
print(f"Archivo '{parquet_file}' exportado con éxito.")
print(f"Tamaño final en disco: {file_size_mb:.2f} MB")

=== 1. CRUCE DE VENTAS CON CATÁLOGO API ===
Registros tras cruzar Ventas + Catálogo: 73457 filas x 14 columnas

=== 2. AGREGACIÓN Y CRUCE CON COMPETENCIA ===
Dimensiones del Dataset Maestro Final: (73457, 18)


,sale_id,date,product_id,units,unit_price,customer_country,channel,api_product_name,category,api_price,discountPercentage,rating,stock,brand,comp_min_price,comp_max_price,comp_avg_price,comp_offers_count
0,S0000001,2026-01-22 11:58:50,31,1,0.79,ES,app,Lemon,groceries,0.79,9.70,3.53,31,NaN,0.69,0.85,0.763333,3.0
1,S0000002,2026-08-08 15:14:19,122,1,262.30,ES,web,iPhone 6,smartphones,299.99,6.69,3.41,60,Apple,279.00,319.99,296.313333,3.0
2,S0000003,2026-03-18 19:45:17,6,1,49.32,PT,app,Calvin Klein CK One,fragrances,49.99,1.89,4.37,29,Calvin Klein,47.90,52.49,50.195000,2.0



=== 3. EXPORTACIÓN Y VALIDACIÓN EN APACHE PARQUET ===
Archivo 'master_sales_market_data.parquet' exportado con éxito.
Tamaño final en disco: 1.82 MB
